# Computer Vision — Practical Examples
**CubicNova Data Science Roadmap · Module 10.05**

This notebook covers image representation, preprocessing, augmentation, convolution, a small CNN, evaluation, and a transfer-learning workflow. It uses synthetic images and scikit-learn's handwritten digits dataset to keep the core examples small.

Run the cells from top to bottom. The notebook's CNN example is educational, not a benchmark model.

## 0. Imports and reproducibility

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device:", device)

## 1. Image representation: a synthetic RGB image

In [ ]:
image = np.zeros((64, 64, 3), dtype=np.uint8)
image[8:40, 8:28, 0] = 255
image[20:55, 30:58, 1] = 180
image[40:60, 10:50, 2] = 220

plt.figure(figsize=(4, 4))
plt.imshow(image)
plt.title(f"RGB image shape: {image.shape}")
plt.axis("off")
plt.show()

print("Shape (H, W, C):", image.shape)
print("Pixel range:", image.min(), "to", image.max())

NumPy/Matplotlib commonly use `(H, W, C)`. PyTorch convolution layers usually expect `(N, C, H, W)` for a batch. Make the layout explicit when converting.

## 2. Image tensor and transforms

In [ ]:
from torchvision.transforms import v2

# Convert HWC NumPy array to CHW tensor
image_t = torch.from_numpy(image).permute(2, 0, 1)
pipeline = v2.Compose([
    v2.Resize((32, 32), antialias=True),
    v2.ToDtype(torch.float32, scale=True),
])
image_small = pipeline(image_t)

print("Before:", image_t.shape, image_t.dtype, image_t.min().item(), image_t.max().item())
print("After:", image_small.shape, image_small.dtype, image_small.min().item(), image_small.max().item())

plt.figure(figsize=(4, 4))
plt.imshow(image_small.permute(1, 2, 0).numpy())
plt.title("Resized image")
plt.axis("off")
plt.show()

Torchvision transform conventions are important: a single tensor image is commonly `(C, H, W)`, while a batch is `(N, C, H, W)`. Float image values are typically scaled to `[0,1]` before normalization. See the official [Torchvision transforms documentation](https://docs.pytorch.org/vision/stable/transforms.html).

## 3. Augmentation preview

In [ ]:
augment = v2.Compose([
    v2.RandomHorizontalFlip(p=1.0),
    v2.RandomRotation(degrees=15),
])
augmented = augment(image_t)

fig, axes = plt.subplots(1, 2, figsize=(7, 3))
axes[0].imshow(image_t.permute(1, 2, 0).numpy())
axes[0].set_title("Original")
axes[1].imshow(augmented.permute(1, 2, 0).numpy())
axes[1].set_title("Augmented")
for ax in axes:
    ax.axis("off")
plt.show()

Augmentation should preserve the label. For some tasks, such as text recognition or left/right orientation, flips may change the label and should not be used blindly.

## 4. Convolution and output shapes

In [ ]:
gray = torch.zeros(1, 1, 32, 32)
gray[:, :, 8:24, 8:24] = 1.0

conv_pad = nn.Conv2d(1, 4, kernel_size=3, stride=1, padding=1)
conv_valid = nn.Conv2d(1, 4, kernel_size=3, stride=1, padding=0)

out_pad = conv_pad(gray)
out_valid = conv_valid(gray)
print("Input:", gray.shape)
print("3x3 conv, padding=1:", out_pad.shape)
print("3x3 conv, padding=0:", out_valid.shape)

plt.figure(figsize=(4, 3))
plt.imshow(out_pad[0, 0].detach().numpy(), cmap="gray")
plt.title("One learned feature map (random initial weights)")
plt.axis("off")
plt.show()

A convolution learns filters that respond to local patterns. Padding changes border handling and output size; channels correspond to learned feature maps.

## 5. Train a small CNN on the digits dataset

The scikit-learn digits dataset contains small 8×8 grayscale handwritten digits. Pixel values are scaled to `[0,1]`. This compact example is useful for learning the CNN training loop; results are not intended as a benchmark.

In [ ]:
digits = load_digits()
X = digits.images.astype(np.float32) / 16.0
y = digits.target.astype(np.int64)

X_train_np, X_test_np, y_train_np, y_test_np = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

# CNN input shape: N, C, H, W
X_train_t = torch.tensor(X_train_np[:, None, :, :], dtype=torch.float32)
X_test_t = torch.tensor(X_test_np[:, None, :, :], dtype=torch.float32)
y_train_t = torch.tensor(y_train_np, dtype=torch.long)
y_test_t = torch.tensor(y_test_np, dtype=torch.long)

train_loader = DataLoader(
    TensorDataset(X_train_t, y_train_t), batch_size=64, shuffle=True
)

class SmallCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 8x8 -> 4x4
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 4x4 -> 2x2
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 2 * 2, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes),  # raw class logits
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model = SmallCNN().to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
history = []

for epoch in range(15):
    model.train()
    total_loss = 0.0
    correct = 0
    seen = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(xb)
        correct += (logits.argmax(dim=1) == yb).sum().item()
        seen += len(xb)
    history.append(total_loss / seen)
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:02d} | loss={history[-1]:.4f} | train_acc={correct/seen:.3f}")

model.eval()
with torch.inference_mode():
    logits = model(X_test_t.to(device)).cpu()
    predictions = logits.argmax(dim=1).numpy()

print("Held-out test accuracy:", round((predictions == y_test_np).mean(), 3))
print(classification_report(y_test_np, predictions, digits=3, zero_division=0))
print("Confusion matrix:\n", confusion_matrix(y_test_np, predictions))

plt.figure(figsize=(7, 3))
plt.plot(history)
plt.xlabel("Epoch")
plt.ylabel("Training cross-entropy")
plt.title("CNN training loss")
plt.grid(True, alpha=0.3)
plt.show()

ConfusionMatrixDisplay.from_predictions(y_test_np, predictions)
plt.title("Digits test confusion matrix")
plt.show()

For `CrossEntropyLoss`, the model returns raw logits with shape `(batch_size, num_classes)` and targets are integer class IDs with dtype `torch.long`. Do not apply softmax before passing logits to this loss.

## 6. Inspect correct and incorrect predictions

In [ ]:
wrong = np.flatnonzero(predictions != y_test_np)
right = np.flatnonzero(predictions == y_test_np)

fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for ax, idx in zip(axes.ravel(), right[:10]):
    ax.imshow(X_test_np[idx], cmap="gray")
    ax.set_title(f"T:{y_test_np[idx]} P:{predictions[idx]}")
    ax.axis("off")
plt.suptitle("Examples classified correctly")
plt.tight_layout()
plt.show()

if len(wrong):
    fig, axes = plt.subplots(1, min(5, len(wrong)), figsize=(10, 2.5))
    axes = np.atleast_1d(axes)
    for ax, idx in zip(axes, wrong[:5]):
        ax.imshow(X_test_np[idx], cmap="gray")
        ax.set_title(f"T:{y_test_np[idx]} P:{predictions[idx]}")
        ax.axis("off")
    plt.suptitle("Examples classified incorrectly")
    plt.tight_layout()
    plt.show()
else:
    print("No test errors in this run; try a smaller model or fewer epochs to inspect errors.")

Error analysis often reveals which classes are visually similar and where more data or improved preprocessing might help.

## 7. Transfer learning: workflow template

Transfer learning reuses a pretrained vision backbone and replaces its final classifier. The following is a template rather than a full download/training run: select a pretrained weights enum appropriate to your installed Torchvision version, use its recommended preprocessing, and verify dataset licensing before training.

```python
from torchvision.models import resnet18, ResNet18_Weights
from torch import nn

weights = ResNet18_Weights.DEFAULT
model = resnet18(weights=weights)

# Freeze pretrained parameters for a feature-extraction baseline
for parameter in model.parameters():
    parameter.requires_grad = False

num_classes = 3  # replace with your dataset's number of classes
model.fc = nn.Linear(model.fc.in_features, num_classes)

# Use the preprocessing associated with the selected weights
preprocess = weights.transforms()

# Train only the new classifier head first.
# Later, optionally unfreeze selected backbone layers and fine-tune with a lower learning rate.
```

Do not assume the example's `num_classes` matches your data. Keep validation and test images separate from model selection.

## 8. Choose the right vision task

- **Classification:** one or more labels for an image.
- **Object detection:** class labels plus bounding boxes.
- **Semantic segmentation:** a class for each pixel.
- **Instance segmentation:** a separate mask for each object instance.
- **Keypoint estimation:** locations of important points.

Torchvision provides datasets, transformations, pretrained models, and utilities for several of these tasks. Explore the [official documentation](https://docs.pytorch.org/vision/stable/).

## 9. Experiment checklist

Change one factor at a time:

1. Number of epochs: `5`, `10`, `20`.
2. Learning rate: `0.0003`, `0.001`, `0.003`.
3. CNN width: reduce or increase the channel counts.
4. Compare a simple baseline with the CNN.
5. Record train loss, held-out metrics, and examples of errors.

For a more rigorous experiment, create a validation split and use it for hyperparameter selection. Keep the test set for final evaluation.

## Key takeaways

- Images have spatial dimensions, channels, and value-range conventions.
- Preprocessing and augmentation should match the task and model.
- CNNs use convolutional filters to learn spatial features.
- Use suitable loss functions and target shapes.
- Evaluate per-class errors and check for leakage, bias, and distribution shift.